### Additional evaluation of model using Alphalens

In [1]:
import pandas as pd
from alphalens import plotting
from alphalens import performance as perf
from alphalens.utils import get_clean_factor_and_forward_returns, rate_of_return, std_conversion
from alphalens.tears import (create_summary_tear_sheet,
                             create_full_tear_sheet)

In [2]:
lookahead = 1
best_predictions = pd.read_hdf('data/predictions.h5', key=f'xgb/train/{lookahead:02}')

In [16]:
trade_prices = pd.read_hdf('data/model_tuning.h5', 'trade_prices/model_selection')
trade_prices = trade_prices.dropna()

In [11]:
factor = best_predictions.iloc[:, :5].mean(1).dropna().tz_localize('UTC', level='date').swaplevel()

In [13]:
factor

date                       asset
2022-11-29 00:00:00+00:00  AAL     -0.003192
2022-11-30 00:00:00+00:00  AAL      0.003457
2022-12-01 00:00:00+00:00  AAL     -0.000768
2022-12-02 00:00:00+00:00  AAL     -0.003464
2022-12-05 00:00:00+00:00  AAL      0.003302
                                      ...   
2023-04-04 00:00:00+00:00  ZION     0.008302
2023-04-05 00:00:00+00:00  ZION     0.010554
2023-04-06 00:00:00+00:00  ZION     0.027619
2023-04-10 00:00:00+00:00  ZION     0.014100
2023-04-11 00:00:00+00:00  ZION     0.007810
Length: 49513, dtype: float32

#### Create AlphaLens Inputs

In [ ]:
factor_data = get_clean_factor_and_forward_returns(factor=factor,
                                                   prices=trade_prices,
                                                   quantiles=5,
                                                   periods=(1, 5, 21))
                                                   #max_loss=1.0)

Dropped 100.0% entries from factor data: 100.0% in forward returns computation and 0.0% in binning phase (set max_loss=0 to see potentially suppressed Exceptions).


MaxLossExceededError: max_loss (35.0%) exceeded 100.0%, consider increasing it.

#### Compute Alphalens metrics

In [6]:
mean_quant_ret_bydate, std_quant_daily = perf.mean_return_by_quantile(
    factor_data,
    by_date=True,
    by_group=False,
    demeaned=True,
    group_adjust=False,
)

ValueError: No objects to concatenate

In [ ]:
factor_returns = perf.factor_returns(factor_data)

In [ ]:
mean_quant_ret, std_quantile = perf.mean_return_by_quantile(factor_data,
                                                            by_group=False,
                                                            demeaned=True)



mean_quant_rateret = mean_quant_ret.apply(rate_of_return, axis=0,
                                          base_period=mean_quant_ret.columns[0])

In [ ]:
mean_quant_ret_bydate, std_quant_daily = perf.mean_return_by_quantile(
    factor_data,
    by_date=True,
    by_group=False,
    demeaned=True,
    group_adjust=False,
)

mean_quant_rateret_bydate = mean_quant_ret_bydate.apply(
    rate_of_return,
    base_period=mean_quant_ret_bydate.columns[0],
)

compstd_quant_daily = std_quant_daily.apply(std_conversion,
                                            base_period=std_quant_daily.columns[0])

alpha_beta = perf.factor_alpha_beta(factor_data,
                                    demeaned=True)

mean_ret_spread_quant, std_spread_quant = perf.compute_mean_returns_spread(
    mean_quant_rateret_bydate,
    factor_data["factor_quantile"].max(),
    factor_data["factor_quantile"].min(),
    std_err=compstd_quant_daily,
)

In [ ]:
mean_ret_spread_quant.mean().mul(10000).to_frame('Mean Period Wise Spread (bps)').join(alpha_beta.T).T

In [ ]:
fig, axes = plt.subplots(ncols=3, figsize=(18, 4))


plotting.plot_quantile_returns_bar(mean_quant_rateret, ax=axes[0])
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=0)
axes[0].set_xlabel('Quantile')

plotting.plot_cumulative_returns_by_quantile(mean_quant_ret_bydate['1D'],
                                             freq=pd.tseries.offsets.BDay(),
                                             period='1D',
                                             ax=axes[1])
axes[1].set_title('Cumulative Return by Quantile (1D Period)')

title = "Cumulative Return - Factor-Weighted Long/Short PF (1D Period)"
plotting.plot_cumulative_returns(factor_returns['1D'],
                                 period='1D',
                                 freq=pd.tseries.offsets.BDay(),
                                 title=title,
                                 ax=axes[2])

fig.suptitle('Alphalens - Validation Set Performance', fontsize=14)
fig.tight_layout()
fig.subplots_adjust(top=.85);

### Summary Tearsheet

In [18]:
create_summary_tear_sheet(factor_data)

ValueError: No objects to concatenate

<Figure size 640x480 with 0 Axes>

In [ ]:
create_full_tear_sheet(factor_data)